# Segmentation

The [Quickstart](../tutorials/quickstart.ipynb) shrank the data by keeping fewer **periods** — a
year of days became a handful of typical days. **Segmentation** is a second,
independent lever that works *inside* each period.

A typical day still has 24 hourly values. Many of those hours are nearly
identical — overnight load barely moves for hours. Segmentation merges runs of
similar consecutive steps into a single **segment** with one value, so a 24-hour
day might be described by just 6 segments. Segments are **variable length**: long
where the profile is flat, short where it changes quickly.

It is easiest to just see it.

In [ ]:
import pandas as pd
import plotly.io as pio

import tsam
from tsam import SegmentConfig

pio.renderers.default = "notebook_connected"

raw = pd.read_csv("../data/testdata.csv", index_col=0, parse_dates=True)
data = raw.loc["2010-01-01":"2010-02-11"]  # six weeks of hourly data

## See it: 24 hours → 6 segments

Aggregate to 6 typical days, but describe each day with **6 segments** instead of
24 hourly steps:

In [ ]:
result = tsam.aggregate(
    data,
    n_clusters=6,
    period_duration="1D",
    segments=SegmentConfig(n_segments=6),
)

Now compare the **original** hourly load with the **reconstructed** series built
from the segmented typical days. The reconstruction is a step function — each flat
step is one segment, held constant across the hours it covers. Where the original
is busy (the daily ramp), segments are short. Where it is calm (overnight), one
long segment spans many hours.

Here is one week — each flat step is a segment. (The plot is interactive, so you can zoom further into a single day.)

In [ ]:
week = slice("2010-01-11", "2010-01-17")
result.plot.compare(columns=["Load"], time_slice=week, color="source")

That is the whole idea: replace 24 hourly values with a few representative steps,
chosen to follow the shape of the data — longer where it is flat, shorter where it
moves.

## What each segment keeps

By default every segment carries the **mean** of the hours it covers. That
representation is the *only* thing you tune about how a segment is described — the
merging algorithm itself is fixed (constrained agglomerative clustering, adjacent
hours only), so there is no segmentation "method" to choose, only `n_segments` and
`representation`.

To keep an actual representative hour instead of an average — or to preserve the
spread — pass a `representation` to `SegmentConfig`. The choices are the same as for
[period representations](representations.ipynb):

In [ ]:
result_medoid = tsam.aggregate(
    data,
    n_clusters=6,
    period_duration="1D",
    segments=SegmentConfig(n_segments=6, representation="medoid"),
)
result_medoid.plot.compare(columns=["Load"], time_slice=week, color="source")

## Why bother: the budget

The two levers multiply. Six typical days at hourly resolution already cut the
six-week series down a lot. Segmenting each day into 6 steps cuts it much further
— for a small accuracy cost.

In [ ]:
plain = tsam.aggregate(data, n_clusters=6, period_duration="1D")

original_steps = len(data)
plain_steps = plain.n_clusters * plain.n_timesteps_per_period  # 6 days x 24 h
seg_steps = result.n_clusters * result.n_segments  # 6 days x 6 segments

pd.DataFrame(
    {
        "time steps": [original_steps, plain_steps, seg_steps],
        "reduction": [
            "—",
            f"{1 - plain_steps / original_steps:.0%}",
            f"{1 - seg_steps / original_steps:.0%}",
        ],
        "mean RMSE": [
            0.0,
            round(float(plain.accuracy.rmse.mean()), 4),
            round(float(result.accuracy.rmse.mean()), 4),
        ],
    },
    index=["original (hourly)", "6 days x 24 h", "6 days x 6 segments"],
)

So you have two dials: how many typical periods, and how fine each one is.
[How small can you go?](tuning.ipynb) searches both for the best combination at a
target size.

## When to segment — and when not to

Both dials shrink the problem, but they throw away different things. Fewer **periods** costs you
variety across days; fewer **segments** costs you detail within a day. Which hurts less depends
entirely on your data and your model.

**Reach for segmentation when:**

- Your profiles have long flat stretches — overnight demand, becalmed wind. Those hours cost you
  time steps and tell you nothing; segmentation is close to free there.
- Your period is long. A 168-hour week has far more redundancy to find than a 24-hour day.
- You are already at the fewest periods your model can tolerate and still need to be smaller.

**Be careful when:**

- **The profile is spiky.** Segmentation merges by similarity, so a short peak neighboured by
  quiet hours is exactly what gets absorbed into a long flat segment. If a peak drives your
  result, check it survived — or pin it down with
  [extreme periods](extreme_periods.ipynb), which operate on periods and so are unaffected by
  this.
- **Intra-period dynamics drive the answer.** Storage that charges and discharges within the day,
  ramp limits, minimum up/down times — these live in the *shape* inside a period, which is the
  thing segmentation compresses. Cutting periods instead leaves each remaining day at full
  resolution.
- **You need a fixed time grid.** Segments are variable-length and differ from one typical period
  to the next, so every downstream sum has to weight by `result.segment_durations`. If your model
  cannot express that, segmentation is not for you.

A rough rule: **cut periods first, segment second.** Periods are usually the cheaper reduction,
and segmentation is the lever you add when periods alone cannot get you small enough.